# 1장 2강: 핵심 모델과 선택 기준 압축 정리

## 실습 목표

- Ames Housing의 `SalePrice` 예측 상황에서 **Linear Regression**이 어떤 역할을 하는지 확인합니다.
- KNN과 K-means가 **거리 기반**이라는 점과 스케일에 민감하다는 특징을 구분합니다.
- Decision Tree의 규칙 기반 분할 특징과 스케일링이 필요 없다는 점을 확인합니다.
- K-means와 PCA의 목적이 각각 **군집화 / 차원축소**임을 구분합니다.
- 데이터의 목적과 특성에 따라 다섯 모델 중 **1차 후보를 선택하는 기준**을 정리합니다.

> 이번 실습은 1장 2강 교안에서 다룬 모델의 **핵심 성격과 선택 기준**만 사용합니다.  
> 교안에서 다루지 않은 성능지표, 교차검증, 하이퍼파라미터 탐색, 앙상블 모델은 사용하지 않습니다.


## 실습 준비

이번 실습에서도 **Ames Housing** 데이터를 사용합니다.

### 사용할 주요 컬럼

- `GrLivArea`: 지상 생활 면적
- `OverallQual`: 주택의 전반적인 품질
- `YearBuilt`: 건축 연도
- `LotArea`: 대지 면적
- `SalePrice`: 주택 판매 가격

먼저 데이터를 불러오고, 사용할 컬럼의 자료형·결측치·기초 통계량을 확인합니다.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("Ames_Housing.csv")

use_cols = ["GrLivArea", "OverallQual", "YearBuilt", "LotArea", "SalePrice"]
data = df[use_cols].copy()

print("데이터 크기:", data.shape)
display(data.head())


In [ ]:
print("[자료형]")
display(data.dtypes.to_frame("dtype"))

print("[결측치 수]")
display(data.isna().sum().to_frame("missing_count"))

print("[기초 통계량]")
display(data.describe())


---

# 필수 1. Linear Regression으로 기준 모델의 성격 확인

## 배경

교안에서 **Linear Regression**은 연속적인 숫자를 예측하는 회귀 모델이며, 각 특성에 가중치를 곱해 더하는 구조라고 배웠습니다. 또한 가중치를 확인할 수 있어 **해석이 쉬운 모델**이라는 특징이 있습니다.

Ames Housing에서 `GrLivArea`, `OverallQual`, `YearBuilt`를 이용해 `SalePrice`를 예측하는 Linear Regression을 만들어봅니다.

## 문제

### 요구사항

1. 입력 데이터 `X`는 `GrLivArea`, `OverallQual`, `YearBuilt`로 구성하고, 정답 `y`는 `SalePrice`로 지정하세요.
2. `train_test_split()`을 이용해 train과 test로 나누세요. `test_size=0.2`, `random_state=42`를 사용합니다.
3. `LinearRegression()` 모델을 만들고 train 데이터로 학습하세요.
4. 학습한 모델의 `intercept_`와 각 특성의 `coef_`를 출력하세요.
5. test 데이터의 앞 5개 행에 대해 예측값을 만들고 실제 `SalePrice`와 나란히 확인하세요.

### 확인 포인트

- `SalePrice`가 연속적인 숫자이므로 Linear Regression을 적용할 수 있는지 확인합니다.
- 계수(`coef_`)를 통해 각 입력 특성이 예측식에서 어떤 방향의 가중치를 갖는지 확인합니다.
- 이번 문제에서는 **모델 성능지표를 계산하지 않습니다.**

### Q&A

- **Q1.** Linear Regression은 회귀와 분류 중 어느 문제에 사용하는 모델인가요?
- **Q2.** Linear Regression이 해석이 쉬운 모델로 설명되는 이유는 무엇인가요?
- **Q3.** `SalePrice` 예측에 Logistic Regression을 사용하지 않는 이유는 무엇인가요?


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# TODO 1. X와 y를 만드세요.


# TODO 2. train / test로 분리하세요.


# TODO 3. Linear Regression을 학습하세요.


# TODO 4. 절편과 계수를 출력하세요.


# TODO 5. test 앞 5개 행의 실제값과 예측값을 비교하세요.


### 필수 1 답변 작성

- **Q1. Linear Regression의 문제 유형:**  
- **Q2. 해석이 쉬운 이유:**  
- **Q3. Logistic Regression을 사용하지 않는 이유:**


---

# 필수 2. K-means와 PCA로 비지도학습 목적 구분

## 배경

교안에서 **K-means**는 정답 레이블 없이 비슷한 데이터를 K개의 군집으로 묶는 군집화 알고리즘이라고 배웠습니다. 거리로 가까운 중심을 찾기 때문에 **특성 스케일에 민감**하여 스케일링을 적용하는 것이 좋습니다.

반면 **PCA**는 많은 특성을 더 적은 수의 새로운 축으로 압축하는 **차원축소** 기법입니다.

같은 Ames Housing의 숫자형 특성을 이용해 두 기법이 결과를 어떻게 다르게 만드는지 확인합니다.

## 문제

### 요구사항

1. `GrLivArea`, `OverallQual`, `YearBuilt`, `LotArea` 네 컬럼으로 `X_unsup`을 만드세요. `SalePrice`는 사용하지 않습니다.
2. `StandardScaler()`로 `X_unsup`의 스케일을 맞추세요.
3. 스케일링한 데이터에 `KMeans(n_clusters=3, random_state=42, n_init=10)`을 적용하고 각 행의 군집 번호를 구하세요.
4. 군집별 데이터 개수를 출력하세요.
5. 원본 네 특성에 `PCA(n_components=2)`를 적용해 2개의 새로운 축으로 변환하세요.
6. PCA 적용 전과 적용 후 데이터의 크기(shape)를 출력하세요.

### 확인 포인트

- K-means에서는 정답 `y` 없이 특성 데이터만 사용합니다.
- K-means의 K는 **군집 개수**를 의미합니다.
- PCA는 데이터를 그룹으로 나누는 것이 아니라 **특성의 개수를 줄이는 것**이 목적입니다.
- 이번 실습에서 K-means의 K를 고르기 위한 Elbow/Silhouette 비교는 진행하지 않습니다.

### Q&A

- **Q1.** K-means는 지도학습과 비지도학습 중 어디에 해당하나요?
- **Q2.** K-means 전에 스케일링을 적용한 이유는 무엇인가요?
- **Q3.** K-means의 K와 KNN의 K는 각각 무엇의 개수인가요?
- **Q4.** PCA를 적용한 뒤 `(행 개수, 특성 개수)`에서 어떤 부분이 바뀌나요?


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# TODO 1. 비지도학습에 사용할 X_unsup을 만드세요.


# TODO 2. StandardScaler로 스케일을 맞추세요.


# TODO 3. K-means로 3개 군집을 만들고 군집 번호를 구하세요.


# TODO 4. 군집별 데이터 개수를 출력하세요.


# TODO 5. PCA로 4개 특성을 2개의 새로운 축으로 변환하세요.


# TODO 6. PCA 적용 전/후 shape를 출력하세요.


### 필수 2 답변 작성

- **Q1. K-means의 학습 방식:**  
- **Q2. K-means 전에 스케일링이 필요한 이유:**  
- **Q3. K-means의 K / KNN의 K 차이:**  
- **Q4. PCA 적용 전후 shape 변화:**


---

# 과제 1. Ames Housing 상황에 맞는 모델 후보 선택

## 배경

이번 강의의 핵심은 모델을 복잡하게 튜닝하는 것이 아니라, **데이터의 목적과 특성을 보고 어떤 모델부터 후보로 볼지 판단하는 것**입니다.

아래 상황은 모두 이번 강의에서 학습한 Linear Regression, Logistic Regression, KNN, Decision Tree, K-means, PCA의 특징만으로 판단할 수 있습니다.

## 문제

각 상황에서 가장 먼저 고려할 모델 또는 기법을 하나 선택하고, **교안에서 배운 선택 기준을 근거로 이유를 한 문장으로 작성하세요.**

### 요구사항

1. `SalePrice`처럼 연속적인 숫자를 예측하면서, 각 특성의 가중치를 확인해 예측 근거도 쉽게 설명하고 싶습니다. 어떤 모델을 먼저 고려해야 하나요?
2. 주택 특성 사이의 관계가 단순한 직선 형태가 아니며, 사람이 이해할 수 있는 `예/아니오` 규칙 형태로 나누고 싶습니다. 어떤 모델을 먼저 고려해야 하나요?
3. 거리 기반으로 주변의 비슷한 주택을 참고하여 예측하려고 합니다. 어떤 모델을 고려할 수 있으며, 사용 전에 무엇을 확인해야 하나요?
4. `SalePrice` 같은 정답 레이블을 사용하지 않고 주택을 비슷한 특성끼리 3개 그룹으로 묶고 싶습니다. 어떤 기법을 사용해야 하나요?
5. 숫자형 특성이 많아 분석하기 어렵기 때문에 핵심 정보를 유지하면서 2개의 새로운 축으로 줄이고 싶습니다. 어떤 기법을 사용해야 하나요?

### 확인 포인트

- 모델 이름만 쓰지 않고 **왜 그 모델을 선택했는지** 교안의 특성과 연결합니다.
- 성능지표 비교, 교차검증, GridSearch 등의 방법은 사용하지 않습니다.
- 과제는 필수 문제에서 학습한 **모델의 성격과 선택 기준을 다시 적용하는 수준**입니다.


### 과제 1 답변 작성

1. **연속값 예측 + 해석 중요:**  
2. **비선형 관계 + 규칙 해석:**  
3. **가까운 이웃 기반 예측:**  
4. **정답 없이 3개 그룹으로 군집화:**  
5. **많은 특성을 2개 축으로 압축:**


---

# 실습 마무리

아래 내용을 한 문장씩 정리해보세요.

1. **Linear Regression은 어떤 상황에서 1차 후보가 될 수 있는가?**
2. **KNN / Decision Tree는 데이터 특성에 따라 어떤 차이가 있는가?**
3. **K-means와 PCA의 목적은 어떻게 다른가?**
4. **처음부터 복잡한 모델을 선택하기보다 후보를 좁힐 때 어떤 기준을 볼 수 있는가?**
